# 📜 Historical Digital Edition Workflow

Interactive walkthrough of the two-stage OCR → NER pipeline for historical German books.

**Stages:**
1. Download images via IIIF
2. OCR — transcribe Fraktur pages into structured JSON
3. NER — annotate named entities
4. Export — interactive HTML edition + CSV

> **Colab users:** Set  in *Secrets* (🔑 icon, left sidebar) before running.

## 0. Install dependencies

In [ ]:
!pip install -q google-genai pillow tqdm python-dotenv

## 1. Environment setup

In [ ]:
import os, sys

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    api_key = userdata.get("GEMINI_API_KEY")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Set GEMINI_API_KEY in Colab Secrets or .env")
print("✅ API key ready")

## 2. Paths & config

In [ ]:
if IN_COLAB:
    IMAGE_FOLDER  = "/content/drive/MyDrive/digital_edition/images"
    OUTPUT_FOLDER = "/content/drive/MyDrive/digital_edition/output"
    REPO_ROOT = "/content/historical-digital-edition"
else:
    IMAGE_FOLDER  = "../images"
    OUTPUT_FOLDER = "../output"
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src import config
print(f"Images : {IMAGE_FOLDER}")
print(f"Output : {OUTPUT_FOLDER}")
print(f"Model  : {config.MODEL_ID}")

## 3. Download images (IIIF)

Download page images from the Bayerische Staatsbibliothek IIIF API.

- **First run (seq 15–102):** Downloads the pages already processed in your initial test.
- **Continuation (seq 103+):** Downloads the remaining unprocessed pages to the end of the book.

Both sets of images go into the same `IMAGE_FOLDER` on your Drive.

In [ ]:
from src.downloader import IIIFDownloader

downloader = IIIFDownloader(
    book_id=config.BOOK_ID,
    output_dir=IMAGE_FOLDER,
    delay=config.DOWNLOAD_DELAY_SECONDS,
)

# --- Choose which pages to download ---
# First run: seq 15-102 (already processed in the initial ipynb test)
# downloaded = downloader.download(start_seq=15, end_seq=102)

# Continuation: seq 103 to end of the book (remaining unprocessed pages)
downloaded = downloader.download(start_seq=103, end_seq=None)

print(f"Downloaded {len(downloaded)} images")

## 4. Test single page

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from google import genai
from src import process_page
from src.pipeline import get_image_files

client = genai.Client(api_key=api_key)
image_files = get_image_files(IMAGE_FOLDER)
print(f"Found {len(image_files)} images")

test_image = image_files[2]  # change index as needed
test_result = process_page(
    client=client,
    image_path=test_image,
    page_number=1,
    entity_types=config.ENTITY_TYPES,
    model_id=config.MODEL_ID,
    thinking_level=config.THINKING_LEVEL,
)

print("
--- OCR (first 500 chars) ---")
print(test_result.ocr_text[:500])
print(f"
--- Entities ({len(test_result.entities)}) ---")
for e in test_result.entities[:15]:
    print(f"  [{e.entity_type}] "{e.text}"")

## 5. Process pages

Run the OCR + NER pipeline. You can process all images in the folder, or
use `start_page` / `end_page` to process only a slice (0-based index into
the sorted file list).

When continuing from a previous run, point this at only the **new** images
(or use the index slice) so you don't reprocess pages you already have.

In [ ]:
from src import process_book

results = process_book(
    client=client,
    image_folder=IMAGE_FOLDER,
    output_folder=OUTPUT_FOLDER,
    entity_types=config.ENTITY_TYPES,
    model_id=config.MODEL_ID,
    thinking_level=config.THINKING_LEVEL,
    start_page=0,
    end_page=None,  # set e.g. to 10 for a quick test
)

total_ents = sum(len(r.entities) for r in results)
print(f"Pages: {len(results)} | Entities: {total_ents}")

## 5b. Load previous results & merge

If you already processed pages 1-90 (seq 15-102) in an earlier run, load
those results from JSON, then merge them with the newly processed pages to
get the complete book.

In [ ]:
from pathlib import Path
from src import load_results_from_json, merge_results

# --- Load previous results ---
# Option A: from the combined JSON file
previous_json = Path(OUTPUT_FOLDER) / "digital_edition_complete.json"

# Option B: from the individual page JSON directory
# previous_json = Path(OUTPUT_FOLDER) / "json"

previous_results = load_results_from_json(previous_json)
print(f"Loaded {len(previous_results)} previously processed pages")

# --- Merge with newly processed pages ---
# `results` comes from the process_book() call in section 5
all_results = merge_results(previous_results, results)
print(f"Combined: {len(all_results)} total pages")
print(f"  Page range: {all_results[0].page_number} - {all_results[-1].page_number}")
print(f"  Total entities: {sum(len(r.entities) for r in all_results)}")

## 6. Generate HTML edition

Creates a single interactive HTML file with all pages, entity highlighting,
and facsimile images.

**Image modes:**
- `image_ref_prefix="images/"` -- reference images by relative path (recommended for Drive)
- `image_folder=IMAGE_FOLDER` -- embed images as base64 (large file, fully self-contained)
- Both `None` -- no facsimile images

In [ ]:
from pathlib import Path
from src import generate_html_edition

# Use all_results (merged) if you ran section 5b, otherwise just results
edition_results = all_results if "all_results" in dir() else results

html_path = generate_html_edition(
    results=edition_results,
    output_path=Path(OUTPUT_FOLDER) / "digital_edition.html",
    title="Historische Digitalausgabe",
    entity_colors=config.ENTITY_COLORS,
    entity_labels=config.ENTITY_LABELS,
    # Reference images on Drive instead of embedding them:
    image_ref_prefix="images/",
    # To embed images as base64 instead, uncomment below and comment out image_ref_prefix:
    # image_folder=IMAGE_FOLDER,
)
print(f"HTML edition: {html_path}")
print(f"Pages: {len(edition_results)}")

if IN_COLAB:
    from google.colab import files
    files.download(str(html_path))

## 7. Export entities to CSV

In [ ]:
import csv
from pathlib import Path

# Use all_results (merged) if available, otherwise just results
edition_results = all_results if "all_results" in dir() else results

csv_path = Path(OUTPUT_FOLDER) / "all_entities.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["page","entity_type","text","start_char","end_char","context"])
    for r in edition_results:
        for e in r.entities:
            writer.writerow([r.page_number,e.entity_type,e.text,e.start_char,e.end_char,e.context])

print(f"CSV saved: {csv_path}")

freq = {}
for r in edition_results:
    for e in r.entities:
        freq[(e.entity_type, e.text)] = freq.get((e.entity_type, e.text), 0) + 1

print("\nTop 20 entities:")
for (etype, text), count in sorted(freq.items(), key=lambda x: -x[1])[:20]:
    print(f"  [{etype}] \"{text}\": {count}x")